# SHAP Analysis — LR & RF Models

Computes SHAP values for the trained Linear Regression (LR) and Random Forest (RF) models
across the **Flow** and **Sigmoid** experiment groups.

| Explainer | Architecture | Notes |
|-----------|-------------|-------|
| `LinearExplainer` | LR | Exact; uses background subset of training data |
| `TreeExplainer` | RF | Exact (tree path-dependent); no background needed |

**Output sections:**
1. Beeswarm plots — feature-level SHAP distribution (top 20 features)
2. Variable-group importance — mean |SHAP| aggregated by variable (gwl, wl, rain, …)
3. Cross-experiment heatmap — normalized importance across all experiments

In [ ]:
import json
import pickle
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

from src.data.preprocessing import (
    create_input_columns,
    create_lagged_columns,
    order_input_arrays,
    split_df_by_years,
    get_xy,
)

warnings.filterwarnings("ignore")
shap.initjs()

## Configuration

Select which experiment JSON files to analyze and sampling parameters for SHAP.

In [ ]:
# Flow experiments (all four variants)
FLOW_JSONS = [ROOT / "experiments" / "24hr_flow_study.json"]

# Sigmoid experiments — measured-only by default (one per lead time)
SIGMOID_JSONS = [
    ROOT / "experiments" / f"{hr}hr_models_measured_sigmoid.json"
    for hr in [3, 6, 12, 24]
]

# Uncomment to also include perfect-prog sigmoid experiments
# SIGMOID_JSONS += [
#     ROOT / "experiments" / f"{hr}hr_models_perf_prog_all_inputs_sigmoid.json"
#     for hr in [3, 6, 12, 24]
# ]

ALL_JSON_FILES = FLOW_JSONS + SIGMOID_JSONS
MODELS_TO_ANALYZE = ["LR", "RF"]

# SHAP sampling
N_LR_BACKGROUND = 500   # background samples for LinearExplainer
N_RF_EXPLAIN    = 500   # test samples to explain with TreeExplainer
RANDOM_SEED     = 42

print("Experiment files to process:")
for f in ALL_JSON_FILES:
    print(f"  {f.name}")

## Helper Functions

In [ ]:
def load_experiment_configs(json_path):
    with open(json_path) as f:
        return json.load(f)["experiments"]


def build_dataset(exp_config):
    """Reconstruct train/test arrays matching the original training pipeline."""
    df = pd.read_csv(ROOT / exp_config["data_file_path"], index_col=0, parse_dates=True)

    # Replicate create_input_dataframe from experiment_runner.py
    df = create_input_columns(df, exp_config["input_specifications"])

    target_col = exp_config["target_column"]
    lead_time  = exp_config["lead_time"]
    df = create_lagged_columns(df, target_col, (lead_time, lead_time))
    df.dropna(inplace=True)

    col_prefixes   = [s["column"] for s in exp_config["input_specifications"]]
    df             = order_input_arrays(df, col_prefixes)
    target_col_fmt = f"{target_col}_t+{lead_time}"

    df_test, df_train = split_df_by_years(df, exp_config["test_years"])
    feature_cols = [c for c in df.columns if c != target_col_fmt]

    X_train, y_train = get_xy(df_train, feature_cols, target_col_fmt)
    X_test,  y_test  = get_xy(df_test,  feature_cols, target_col_fmt)

    return X_train, y_train, X_test, y_test, list(feature_cols)


def load_sklearn_model(exp_name, arch):
    model_path = ROOT / "results" / exp_name / arch / "models" / "hypermodel.pkl"
    with open(model_path, "rb") as f:
        return pickle.load(f)


def compute_shap(model, arch, X_train, X_test):
    """Return (shap_values, X_explain) using the arch-appropriate explainer."""
    rng = np.random.default_rng(RANDOM_SEED)

    if arch == "LR":
        bg_idx = rng.choice(len(X_train), min(N_LR_BACKGROUND, len(X_train)), replace=False)
        explainer = shap.LinearExplainer(model, X_train[bg_idx])
        shap_vals = explainer.shap_values(X_test)
        return shap_vals, X_test

    elif arch == "RF":
        ex_idx = rng.choice(len(X_test), min(N_RF_EXPLAIN, len(X_test)), replace=False)
        X_explain = X_test[ex_idx]
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_explain)
        return shap_vals, X_explain

    raise ValueError(f"Unsupported architecture: {arch}")


def group_shap_importance(shap_values, feature_names):
    """Aggregate mean |SHAP| by variable prefix (e.g., 'gwl', 'wl', 'flow')."""
    # Preserve insertion order
    seen, prefixes = set(), []
    for name in feature_names:
        prefix = name.split("_t")[0]
        if prefix not in seen:
            prefixes.append(prefix)
            seen.add(prefix)

    importance = {}
    for prefix in prefixes:
        idx = [i for i, n in enumerate(feature_names) if n.split("_t")[0] == prefix]
        importance[prefix] = float(np.abs(shap_values[:, idx]).mean())
    return importance

## Compute SHAP Values

Loops over all configured experiments and model architectures. Results are stored in
`shap_results` keyed by `(experiment_name, model_arch)` so plots can be regenerated
without recomputing.

In [ ]:
shap_results = {}  # {(exp_name, arch): {"shap_values", "X_explain", "feature_names"}}

for json_file in ALL_JSON_FILES:
    for exp_config in load_experiment_configs(json_file):
        exp_name = exp_config["experiment_name"]
        print(f"\n{'='*60}")
        print(f"Experiment: {exp_name}")

        X_train, y_train, X_test, y_test, feature_names = build_dataset(exp_config)
        print(f"  Features: {len(feature_names)} | Train: {len(X_train):,} | Test: {len(X_test):,}")

        for arch in MODELS_TO_ANALYZE:
            model_path = ROOT / "results" / exp_name / arch / "models" / "hypermodel.pkl"
            if not model_path.exists():
                print(f"  [{arch}] model not found — skipping")
                continue

            print(f"  [{arch}] computing SHAP...", end=" ", flush=True)
            model = load_sklearn_model(exp_name, arch)
            shap_vals, X_explain = compute_shap(model, arch, X_train, X_test)

            shap_results[(exp_name, arch)] = {
                "shap_values":  shap_vals,
                "X_explain":    X_explain,
                "feature_names": feature_names,
            }
            print(f"done  ({len(X_explain)} samples explained)")

print("\nAll SHAP computations complete.")

## Beeswarm Plots

Top-20 features by mean |SHAP value|. Colour = feature value (blue = low, red = high);
x-axis = SHAP impact on model output (positive → pushes GWL prediction higher).

In [ ]:
TOP_N_FEATURES = 20

for (exp_name, arch), data in shap_results.items():
    shap.summary_plot(
        data["shap_values"],
        data["X_explain"],
        feature_names=data["feature_names"],
        max_display=TOP_N_FEATURES,
        show=False,
        plot_size=(10, 6),
    )
    plt.title(f"{exp_name} — {arch}", fontsize=11, pad=10)
    plt.tight_layout()
    plt.show()

## Variable-Group Importance

Mean |SHAP| summed across all lag hours within each variable group, displayed as a
horizontal bar chart. Reveals which physical drivers matter most for each model and
forecast horizon.

In [ ]:
for (exp_name, arch), data in shap_results.items():
    importance = group_shap_importance(data["shap_values"], data["feature_names"])
    variables  = list(importance.keys())
    values     = [importance[v] for v in variables]

    fig, ax = plt.subplots(figsize=(7, max(3, len(variables) * 0.5)))
    bars = ax.barh(variables, values, color="steelblue", edgecolor="white")
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title(f"{exp_name} — {arch}: Variable-Group Importance")
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

## Cross-Experiment Comparison Heatmap

Normalized mean |SHAP| per variable group (rows = experiment × model, columns = variable).
Each row is divided by its own maximum so experiments with different prediction scales are
comparable. Separate heatmaps are produced for LR and RF.

In [ ]:
# Collect all variable groups seen across experiments
all_vars: list[str] = []
seen_vars: set[str] = set()
for data in shap_results.values():
    for name in data["feature_names"]:
        prefix = name.split("_t")[0]
        if prefix not in seen_vars:
            all_vars.append(prefix)
            seen_vars.add(prefix)

# Build importance DataFrame
records = []
for (exp_name, arch), data in shap_results.items():
    importance = group_shap_importance(data["shap_values"], data["feature_names"])
    row = {"experiment": exp_name, "model": arch}
    row.update({v: importance.get(v, 0.0) for v in all_vars})
    records.append(row)

df_imp = pd.DataFrame(records).set_index(["experiment", "model"])

for arch in MODELS_TO_ANALYZE:
    df_arch = df_imp.xs(arch, level="model") if arch in df_imp.index.get_level_values("model") else None
    if df_arch is None or df_arch.empty:
        continue

    # Normalize each row to [0, 1]
    df_norm = df_arch.div(df_arch.max(axis=1).replace(0, 1), axis=0)

    fig, ax = plt.subplots(figsize=(max(8, len(all_vars) * 1.1), max(4, len(df_norm) * 0.5)))
    im = ax.imshow(df_norm.values, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label="Normalized mean |SHAP|")

    ax.set_xticks(range(len(df_norm.columns)))
    ax.set_xticklabels(df_norm.columns, rotation=40, ha="right", fontsize=9)
    ax.set_yticks(range(len(df_norm)))
    ax.set_yticklabels(df_norm.index, fontsize=8)
    ax.set_title(f"{arch} — Variable-Group Importance (normalized per experiment)", fontsize=11)

    # Annotate cells
    for i in range(len(df_norm)):
        for j in range(len(df_norm.columns)):
            val = df_norm.iloc[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=7, color="black" if val < 0.7 else "white")

    plt.tight_layout()
    plt.show()

## Raw Importance Table

In [ ]:
df_imp.round(6)

## Temporal Importance Within a Variable (Optional)

For a single experiment and model, plot mean |SHAP| at each individual lag hour
for every variable. Shows *when* in the observation window each driver matters most.

In [ ]:
# Configure which experiment/model to inspect
INSPECT_EXP  = "24hr_flow_all_inputs"
INSPECT_ARCH = "LR"

key = (INSPECT_EXP, INSPECT_ARCH)
if key not in shap_results:
    print(f"No results found for {key}. Adjust INSPECT_EXP / INSPECT_ARCH above.")
else:
    data         = shap_results[key]
    shap_vals    = data["shap_values"]
    feature_names = data["feature_names"]

    # Group features by variable prefix
    seen, prefixes = set(), []
    for name in feature_names:
        prefix = name.split("_t")[0]
        if prefix not in seen:
            prefixes.append(prefix)
            seen.add(prefix)

    n_vars = len(prefixes)
    fig, axes = plt.subplots(n_vars, 1, figsize=(12, n_vars * 2.5), sharex=False)
    if n_vars == 1:
        axes = [axes]

    for ax, prefix in zip(axes, prefixes):
        idx   = [i for i, n in enumerate(feature_names) if n.split("_t")[0] == prefix]
        names = [feature_names[i] for i in idx]
        vals  = np.abs(shap_vals[:, idx]).mean(axis=0)

        # Extract lag hour for x-axis labels
        def _lag(name):
            if "_t" in name:
                return int(name.split("_t")[-1].replace("+", ""))
            return 0

        lags = [_lag(n) for n in names]

        ax.bar(lags, vals, width=0.8, color="steelblue", edgecolor="white")
        ax.set_ylabel("Mean |SHAP|", fontsize=8)
        ax.set_title(prefix, fontsize=9, loc="left")
        ax.axvline(0, color="red", linewidth=0.8, linestyle="--", label="t=0")
        ax.spines[["top", "right"]].set_visible(False)

    axes[-1].set_xlabel("Lag hour (negative = past, positive = future)")
    fig.suptitle(f"Temporal Importance — {INSPECT_EXP} | {INSPECT_ARCH}", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()